In [ ]:
# Copyright 2022 The ML Notebooks Authors.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Linear Regression

<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/mahalrs/ml-notebooks/blob/main/linear/linear_regression.ipynb"><img src="https://raw.githubusercontent.com/mahalrs/ml-notebooks/main/assets/icons/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/mahalrs/ml-notebooks/blob/main/linear/linear_regression.ipynb"><img src="https://raw.githubusercontent.com/mahalrs/ml-notebooks/main/assets/icons/github_mark_32px.png" />View source on GitHub</a>
  </td>
</table>

This notebook uses the classic Auto MPG dataset and demonstrates how to build a linear regression model to predict the fuel efficiency of the late-1970s and early 1980s automobiles.

## Setup

In [ ]:
import pandas as pd

from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error

## The Auto MPG dataset

The dataset is available from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/).

### Get the data

First download and import the dataset using pandas.

In [ ]:
path = 'http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'
column_names = [
    'MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight', 'Acceleration',
    'Model Year', 'Origin'
]

raw_dataset = pd.read_csv(path,
                          names=column_names,
                          na_values='?',
                          comment='\t',
                          sep=' ',
                          skipinitialspace=True)

In [ ]:
df = raw_dataset.copy()
df.head()

### Clean the data

Check if the dataset contains unknown values.

In [ ]:
df.isna().sum()

Drop rows with unknown values.

In [ ]:
df = df.dropna()

The `"Origin"` column is categorical, not numeric. So, the next step is to one-hot encode the values.

In [ ]:
df_transformed = pd.get_dummies(df,
                                prefix=['Origin'],
                                columns=['Origin'],
                                dummy_na=False)
df_transformed.head()

### Split features from labels

In [ ]:
X = df_transformed.drop(columns=['MPG'])
y = df_transformed['MPG']

### Split the data into development and test sets

Now split the dataset into a development set and a test set. You will use the test set in the final evaluation of your models.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(X,
                                                y,
                                                test_size=0.2,
                                                random_state=42)


### Normalize the data

In [ ]:
scaler = StandardScaler()
X_dev = scaler.fit_transform(X_dev)
X_test = scaler.transform(X_test)

## Build a linear regression model

In [ ]:
model = LinearRegression()

## Train and evaluate your model

Evaluate model performance using cross-validation.

In [ ]:
scores = cross_val_score(model,
                         X_dev,
                         y_dev,
                         scoring='r2',
                         cv=5,
                         error_score='raise')

print('R^2 scores:\n', scores)
print('Mean R^2:\n', scores.mean())

Fit a linear regression model on the development data.

In [ ]:
model.fit(X_dev, y_dev)

print('Coefficients:\n', model.coef_)
print('Intercept:\n', model.intercept_)

Evaluate your model on the test data.

In [ ]:
score = model.score(X_test, y_test)
print('R^2:\n', score)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print('MSE:\n', mse)

fig, ax = plt.subplots()
sns.scatterplot(x=X_test[:, 1], y=y_test, color='r', label='Actual', ax=ax)
sns.regplot(x=X_test[:, 1],
            y=y_pred,
            color='b',
            line_kws={'color': 'k'},
            label='Predicted',
            ax=ax)
ax.legend()
plt.show()

## Conclusion

You have trained a simple linear regression model using scikit-learn.